In [10]:
# IrisPathQ Route Optimizer v0.2

#Classical preprocessing for quantum route optimization

#- Fuel and Distance calculations
#- Cost matrix construction

import subprocess
import os

print("IrisPathQ Route Optimizer v0.2")
print("Setting up C compilation environment...\n")

# gcc verification
result = subprocess.run(["gcc", "--version"], capture_output=True, text=True)
print(result.stdout.split('\n')[0])
print("\nReady for next step")

IrisPathQ Route Optimizer v0.2
Setting up C compilation environment...

gcc (GCC) 15.1.0

Ready for next step


In [11]:
%%writefile data_structures.h
/**
 * IrisPathQ Route Optimizer
 * Data structures for flights, routes, and optimization
*/

#ifndef DATA_STRUCTURES_H
#define DATA_STRUCTURES_H

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

#define MAX_FLIGHTS 500
#define MAX_WAYPOINTS 1000
#define MAX_ROUTES_PER_FLIGHT 10
#define MAX_WAYPOINTS_PER_ROUTE 50
#define EARTH_RADIUS_KM 6371.0
#define M_PI 3.14159265358979323846

typedef struct {
    char id[10];
    char name[50];
    double latitude;
    double longitude;
} Waypoint;

typedef struct {
    double latitude;
    double longitude;
    double wind_speed;
    double wind_direction;
    char weather_type[20];
} Weather;

typedef struct {
    char type[10];
    double cruise_speed;
    double fuel_burn_rate;
    double max_range;
} AircraftType;

typedef struct {
    char flight_id[10];
    char origin[10];
    char destination[10];
    char departure_time[10];
    AircraftType aircraft;
} Flight;

typedef struct {
    int waypoint_indices[MAX_WAYPOINTS_PER_ROUTE];
    int num_waypoints;
    double total_distance;
    double fuel_cost;
    double time_cost;
    int conflict_count;
} Route;

typedef struct {
    Flight flights[MAX_FLIGHTS];
    int num_flights;
    
    Waypoint waypoints[MAX_WAYPOINTS];
    int num_waypoints;
    
    Weather weather_cells[100];
    int num_weather_cells;
    
    Route routes[MAX_FLIGHTS][MAX_ROUTES_PER_FLIGHT];
    int num_routes_per_flight[MAX_FLIGHTS];
} ProblemInstance;

double calculate_distance(double lat1, double lon1, double lat2, double lon2);
double calculate_fuel(Route *route, AircraftType *aircraft);
void print_route(Route *route, Waypoint *waypoints);
int is_in_hazard(double lat, double lon, Weather *weather_cells, int num_cells);

int load_flights(const char *filename, ProblemInstance *problem);
int load_waypoints(const char *filename, ProblemInstance *problem);
int load_weather(const char *filename, ProblemInstance *problem);
int generate_alternative_routes(ProblemInstance *problem, int flight_idx, int num_routes);
void build_cost_matrix(ProblemInstance *problem, double **cost_matrix, int *matrix_size);
void export_cost_matrix(double *cost_matrix, int matrix_size, const char *filename);
void export_full_matrix(double *cost_matrix, int matrix_size, const char *filename);

#endif

Writing data_structures.h


In [12]:
%%writefile calculations.c
/**
 * IrisPathQ Route Optimizer
 * Distance and fuel calculations
*/

#include "data_structures.h"

double calculate_distance(double lat1, double lon1, double lat2, double lon2) {
    //degree to radian conversion
    double lat1_rad = lat1 * M_PI / 180.0;
    double lat2_rad = lat2 * M_PI / 180.0;
    double delta_lat = (lat2 - lat1) * M_PI / 180.0;
    double delta_lon = (lon2 - lon1) * M_PI / 180.0;

    //haversineformula 
    double a = sin(delta_lat / 2.0) * sin(delta_lat / 2.0) +
               cos(lat1_rad) * cos(lat2_rad) *
               sin(delta_lon / 2.0) * sin(delta_lon / 2.0);
    
    double c = 2.0 * atan2(sqrt(a), sqrt(1.0 - a));
    double distance_km = EARTH_RADIUS_KM * c;
    
    return distance_km / 1.852;
}

double calculate_fuel(Route *route, AircraftType *aircraft) {
    double time_hours = route->total_distance / aircraft->cruise_speed;
    return time_hours * aircraft->fuel_burn_rate;
}

// For conflicts due to weather, but rigth now no conflicts-- weather.csv has no thunderstorm as input
int is_in_hazard(double lat, double lon, Weather *weather_cells, int num_cells) {
    for (int i = 0; i < num_cells; i++) {
        if (strcmp(weather_cells[i].weather_type, "THUNDERSTORM") == 0) {
            double dist = calculate_distance(lat, lon, 
                                            weather_cells[i].latitude, 
                                            weather_cells[i].longitude);
            if (dist < 50.0) {
                return 1;
            }
        }
    }
    return 0;
}

void print_route(Route *route, Waypoint *waypoints) {
    printf("Route: ");
    for (int i = 0; i < route->num_waypoints; i++) {
        printf("%s ", waypoints[route->waypoint_indices[i]].id);
        if (i < route->num_waypoints - 1) printf("-> ");
    }
    printf("\n");
    printf("Distance: %.2f nm, Fuel: %.2f kg, Time: %.2f hrs\n",
           route->total_distance, route->fuel_cost, route->time_cost);
}

Writing calculations.c


In [20]:
import subprocess

print("Testing C compilation...\n")

result = subprocess.run(["gcc","-c", "calculations.c", "-o", "calculations.o"], capture_output = True ,text = True)

if result.returncode == 0: 
 print("Compilation works\n")
else:
 print("compilation failed\n")
 print(result.stderr)



Testing C compilation...

Compilation works

